# CricketIQ · M2 — GRU sequence win-probability model

Trains a **causal (unidirectional) GRU** that reads the chase ball-by-ball and
emits P(win) at every ball. Uses the **same 11 features as B1 (LightGBM)** so
this is a clean apples-to-apples test: *does sequential memory beat treating each
ball independently?*

- **Input:** `sequences.npz` (built locally by `build_sequences.py`) uploaded as a Kaggle Dataset.
- **Split:** temporal, by season — train ≤2023 · val 2024 · test ≥2025 (identical to B1).
- **Runtime:** GPU T4 (Settings → Accelerator → **GPU T4 x2**, we use one).
- **Output:** test predictions + weights saved to `/kaggle/working/`.

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| torch", torch.__version__)

device: cuda | torch 2.10.0+cu128


## 1 · Load the sequences

`build_sequences.py` saved four arrays. If `np.load` complains about a key,
run `print(data.files)` to see the real names and adjust below.

In [2]:
import glob
# auto-find sequences.npz under /kaggle/input (robust to the dataset slug)
hits = glob.glob("/kaggle/input/**/sequences.npz", recursive=True)
assert hits, "sequences.npz not found — check the dataset is attached in the Input panel"
path = hits[0]
print("loading:", path)

data = np.load(path)
print("arrays in file:", data.files)

X      = data["X"].astype(np.float32)   # (N, T, F)  padded with 0
mask   = data["mask"].astype(bool)      # (N, T)     True = a real ball
y      = data["y"].astype(np.float32)   # (N,)       chase_won (per match)
season = data["season"].astype(int)     # (N,)       season year

N, T, F = X.shape
FEATURES = ["over","innings_runs","wickets_in_hand","balls_remaining",
            "runs_needed","target","current_rr","required_rr","rr_diff",
            "runs_last30","wkts_last30"]
assert F == len(FEATURES), (F, len(FEATURES))
print(f"N={N}  T={T}  F={F}  |  real balls total = {int(mask.sum()):,}")

loading: /kaggle/input/datasets/anurag9557/cricketiq-sequences/sequences.npz
arrays in file: ['X', 'mask', 'y', 'season', 'feature_names', 'match_ids']
N=6977  T=139  F=11  |  real balls total = 782,542


## 2 · Temporal split by season

Same rule as every other model in the ladder. Splitting **by match** (not by
ball) is what keeps the test set leak-free: no ball from a 2025 match ever
influences training.

In [3]:
train_idx = np.where(season <= 2023)[0]
val_idx   = np.where(season == 2024)[0]
test_idx  = np.where(season >= 2025)[0]
print(f"matches  train {len(train_idx)}  val {len(val_idx)}  test {len(test_idx)}")
print(f"balls    train {int(mask[train_idx].sum()):,}  "
      f"val {int(mask[val_idx].sum()):,}  test {int(mask[test_idx].sum()):,}")

matches  train 4805  val 843  test 1329
balls    train 545,310  val 91,247  test 145,985


## 3 · Standardize features — fit on TRAIN real balls only

The 11 features live on wildly different scales (`target`≈160, `over`≈10,
`wickets_in_hand`≈5). A GRU is scale-sensitive, so we z-score them. Crucially the
mean/std are computed **only over real balls in the training matches** — computing
them over val/test (or over padded zeros) would leak.

In [4]:
flat = X[train_idx][mask[train_idx]]        # (num_real_train_balls, F)
mu = flat.mean(axis=0)
sd = flat.std(axis=0) + 1e-8
for name, m, s in zip(FEATURES, mu, sd):
    print(f"  {name:16s} mean {m:8.2f}  std {s:7.2f}")

Xn = ((X - mu) / sd).astype(np.float32)
Xn = Xn * mask[..., None]                    # re-zero padded steps (tidy)

  over             mean     9.80  std    5.51
  innings_runs     mean    71.16  std   45.98
  wickets_in_hand  mean     7.33  std    2.27
  balls_remaining  mean    64.01  std   33.11
  runs_needed      mean    92.72  std   50.23
  target           mean   163.90  std   32.47
  current_rr       mean     7.47  std    2.49
  required_rr      mean     9.85  std    5.81
  rr_diff          mean     2.38  std    6.50
  runs_last30      mean    32.51  std   14.54
  wkts_last30      mean     1.27  std    1.15


## 4 · DataLoaders

Each item is one **match**: its ball tensor `(T, F)`, its `mask (T,)`, and its
single outcome label `y`. The label is broadcast to every ball inside the loss —
at ball *t* we ask "does this chase eventually win?", exactly the B1 framing.

In [5]:
def make_loader(idx, batch_size, shuffle):
    ds = TensorDataset(
        torch.from_numpy(Xn[idx]),
        torch.from_numpy(mask[idx]),
        torch.from_numpy(y[idx]),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(train_idx, 128, True)
val_loader   = make_loader(val_idx,   256, False)
test_loader  = make_loader(test_idx,  256, False)

## 5 · The model — a **causal** GRU

`bidirectional=False` is not a default we forgot to change — it is the whole
point. Win-probability at ball *t* may use only balls 1..*t*. A bidirectional
GRU would read future balls and leak the outcome. A unidirectional GRU's output
at step *t* depends solely on 1..*t*, so it's causal by construction — and any
padded steps after the last real ball can't affect the real outputs.

In [6]:
class WinProbGRU(nn.Module):
    def __init__(self, n_features, hidden=64, layers=1, dropout=0.0):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_features, hidden_size=hidden, num_layers=layers,
            batch_first=True, bidirectional=False,      # <-- causal: no peeking ahead
            dropout=dropout if layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden, 1)                # per-step logit
    def forward(self, x):
        out, _ = self.gru(x)                            # (B, T, hidden)
        return self.head(out).squeeze(-1)               # (B, T) logits

model = WinProbGRU(F, hidden=64, layers=1).to(device)
print(model)
print("trainable params:", sum(p.numel() for p in model.parameters()))

WinProbGRU(
  (gru): GRU(11, 64, batch_first=True)
  (head): Linear(in_features=64, out_features=1, bias=True)
)
trainable params: 14849


## 6 · Masked BCE at every ball

We supervise **every real ball**, not just the last one — that's what makes it a
per-ball win-prob model. Padded balls are dropped from the loss via the mask.

In [7]:
bce = nn.BCEWithLogitsLoss(reduction="none")

def masked_loss(logit, m, yb):
    target = yb[:, None].expand_as(logit)   # broadcast match label to every ball
    loss = bce(logit, target)               # (B, T)
    return loss[m].mean()                   # keep only real balls

## 7 · Train with early stopping on val

Early stop watches **val log-loss** (the metric we actually care about for a
probabilistic model) and restores the best weights.

In [8]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
EPOCHS, PATIENCE = 40, 6
best_val, best_state, bad = float("inf"), None, 0

def val_logloss(loader):
    model.eval(); tot = n = 0
    with torch.no_grad():
        for xb, m, yb in loader:
            xb, m, yb = xb.to(device), m.to(device), yb.to(device)
            k = int(m.sum().item())
            tot += masked_loss(model(xb), m, yb).item() * k
            n += k
    return tot / n

for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, m, yb in train_loader:
        xb, m, yb = xb.to(device), m.to(device), yb.to(device)
        opt.zero_grad()
        loss = masked_loss(model(xb), m, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
    vl = val_logloss(val_loader)
    flag = ""
    if vl < best_val - 1e-4:
        best_val, bad = vl, 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        flag = "  <- best"
    else:
        bad += 1
    print(f"epoch {epoch:2d}  val_logloss {vl:.4f}{flag}")
    if bad >= PATIENCE:
        print(f"early stop (no val improvement for {PATIENCE} epochs)")
        break

model.load_state_dict(best_state)
print(f"restored best val_logloss {best_val:.4f}")

epoch  1  val_logloss 0.4048  <- best
epoch  2  val_logloss 0.3849  <- best
epoch  3  val_logloss 0.3799  <- best
epoch  4  val_logloss 0.3802
epoch  5  val_logloss 0.3822
epoch  6  val_logloss 0.3743  <- best
epoch  7  val_logloss 0.3741  <- best
epoch  8  val_logloss 0.3732  <- best
epoch  9  val_logloss 0.3754
epoch 10  val_logloss 0.3717  <- best
epoch 11  val_logloss 0.3722
epoch 12  val_logloss 0.3719
epoch 13  val_logloss 0.3715  <- best
epoch 14  val_logloss 0.3718
epoch 15  val_logloss 0.3733
epoch 16  val_logloss 0.3687  <- best
epoch 17  val_logloss 0.3727
epoch 18  val_logloss 0.3699
epoch 19  val_logloss 0.3692
epoch 20  val_logloss 0.3701
epoch 21  val_logloss 0.3682  <- best
epoch 22  val_logloss 0.3693
epoch 23  val_logloss 0.3705
epoch 24  val_logloss 0.3713
epoch 25  val_logloss 0.3720
epoch 26  val_logloss 0.3724
epoch 27  val_logloss 0.3674  <- best
epoch 28  val_logloss 0.3713
epoch 29  val_logloss 0.3683
epoch 30  val_logloss 0.3708
epoch 31  val_logloss 0.3682
ep

## 8 · Collect per-ball test predictions

We flatten to one row per real ball so the numbers are directly comparable to
B1's per-ball test metrics. **`test balls` here should equal B1's test `n`** — a
free cross-check that we're scoring the same set of deliveries.

In [9]:
def collect(loader):
    model.eval(); ps, ys = [], []
    with torch.no_grad():
        for xb, m, yb in loader:
            p = torch.sigmoid(model(xb.to(device))).cpu().numpy()   # (B, T)
            m = m.numpy(); yb = yb.numpy()
            for i in range(len(yb)):
                v = m[i]
                ps.append(p[i][v])
                ys.append(np.full(int(v.sum()), yb[i], dtype=np.float32))
    return np.concatenate(ps), np.concatenate(ys)

p_test, y_test = collect(test_loader)
print("test balls:", len(y_test))

test balls: 145985


## 9 · Metrics — inlined (same definitions as the `cricketiq` package)

Brier · log-loss · AUC · ECE. Identical formulas to `cricketiq/eval/metrics.py`
so the M2 row drops straight into the ladder next to B0/B1.

In [10]:
def brier(y, p):
    return float(np.mean((p - y) ** 2))

def logloss(y, p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

def ece(y, p, n_bins=10):
    ids = np.minimum((p * n_bins).astype(int), n_bins - 1)
    e = 0.0
    for b in range(n_bins):
        sel = ids == b
        if sel.sum():
            e += (sel.sum() / len(p)) * abs(y[sel].mean() - p[sel].mean())
    return float(e)

print("=" * 52)
print("M2  GRU (causal, hidden=64)")
print(f"  Brier    {brier(y_test, p_test):.4f}")
print(f"  log-loss {logloss(y_test, p_test):.4f}")
print(f"  AUC      {roc_auc_score(y_test, p_test):.4f}")
print(f"  ECE      {ece(y_test, p_test):.4f}")
print(f"  n        {len(y_test):,}")
print("=" * 52)
print("compare vs B1 (LightGBM): Brier 0.1144 · logloss 0.3520 · AUC 0.9226 · ECE 0.0261")

M2  GRU (causal, hidden=64)
  Brier    0.1142
  log-loss 0.3515
  AUC      0.9231
  ECE      0.0295
  n        145,985
compare vs B1 (LightGBM): Brier 0.1144 · logloss 0.3520 · AUC 0.9226 · ECE 0.0261


## 10 · Save outputs to bring back

Download these two from the **Output** tab and drop the predictions into your
repo (e.g. `data/processed/m2_gru_test_preds.npz`) so you can regenerate plots /
per-phase splits locally without a GPU.

In [11]:
np.savez("/kaggle/working/m2_gru_test_preds.npz", p=p_test, y=y_test)
torch.save(model.state_dict(), "/kaggle/working/m2_gru.pt")
print("saved: m2_gru_test_preds.npz, m2_gru.pt")

saved: m2_gru_test_preds.npz, m2_gru.pt
